<a href="https://colab.research.google.com/github/Kwasi-Dankwa/rag_project_insurance/blob/main/MeridianHealth_Context_Aware_RAG_MLS_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><p float="center">
  <img src="https://vectorseek.com/wp-content/uploads/2023/08/Massachusetts-Institute-of-Technology-MIT-Logo-Vector.svg-.png" width="360"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200"/>
</p></center>

<center><font size=10>AI and Data Science</font></center>

<center><font size=6>Week 11 - Building Context-Aware AI Workflows</font></center>

<center><p float="center">
  <img src="https://images.pexels.com/photos/7163956/pexels-photo-7163956.jpeg" alt="health-insurance" width="640"/>
</p></center>

<center><font size=6>MeridianHealth: Building a Context-Aware RAG Assistant for Insurance Policy Queries</center></font>

# **1. Index**

This case study follows the end-to-end journey of a Retrieval-Augmented Generation (RAG) assistant, from MeridianHealth policy documents to a reusable interactive inference application.

## 1.1. Sections

- **2. Learning Objectives**
- **3. Problem Statement**
  - 3.1. Business Context
  - 3.2. Problem Framing
  - 3.3. Key Business Challenges
    - 3.3.1. Finding the Right Information
    - 3.3.2. Preserving Policy Meaning
    - 3.3.3. Grounding the Answer
    - 3.3.4. Traceability
  - 3.4. Objective
    - 3.4.1. Implementation Objectives
  - 3.5. Success Criteria
- **4. Installing and Importing the Necessary Libraries**
- **5. Data Understanding**
  - 5.1. Policy Documents
  - 5.2. Supporting Reference: FAQ
- **6. Past Approach**
- **7. RAG Workflow Overview**
- **8. Document Loading and Structuring**
  - 8.1. Converting Policy PDFs with Docling
  - 8.2. Inspecting the Extracted Markdown
  - 8.3. Inspecting Extracted Tables
- **9. Chunking**
  - 9.1. Chunking and Chunk Overlap
  - 9.2. Choosing the Embedding Model
  - 9.3. Defining the RecursiveCharacterTextSplitter
  - 9.4. Why These Parameters?
- **10. Vector Database**
  - 10.1. Loading the Embedding Model
  - 10.2. Creating the FAISS Vector Database
- **11. Retrieval System**
  - 11.1. What Should We Inspect Here?
  - 11.2. Building a Reusable Retrieval Function
- **12. Generation System**
  - 12.1. Configuration and LLM Setup
    - 12.1.1. `config.json` template
  - 12.2. Defining the Generation Prompt
- **13. Retrieval-Augmented Generation System**
  - 13.1. Testing the RAG Pipeline
  - 13.2. Reloading the Vector Database
- **14. Building and Running the Streamlit Inference App in Local VS Code**
  - 14.1. Writing `app.py` with Colab Magic
  - 14.2. Preparing the App and Vector Database for Download
  - 14.3. Downloading the App Bundle from Colab
  - 14.4. Creating the Local Project Folder
  - 14.5. Opening the Project in VS Code
  - 14.6. Creating a Virtual Environment
  - 14.7. Installing the Required Packages
  - 14.8. Creating `config.json`
  - 14.9. Running the Streamlit Application
  - 14.10. Testing the Application with Policy Questions
  - 14.11. Important Notes
- **15. Business Insights and Recommendations**
  - 15.1. Business Insights
  - 15.2. Business Recommendations

> **Scope note:** This MLS focuses on policy document preparation, indexing, retrieval, grounded generation, and inference packaging. Detailed prompt optimisation and systematic evaluation are intentionally outside the scope of this notebook and are covered in dedicated later learning content.

# **2. Learning Objectives**

By the end of this session, you will be able to:

- Understand how to turn policy PDFs into a searchable knowledge base by chunking documents in a way that keeps sections, tables, and clauses intact.
- Learn how to choose and set up an embedding model suited to a specific domain, and use it to build a vector database for semantic search.
- Learn how to build a retrieval and generation flow that pulls relevant policy context and generates answers grounded only in that context.
- Understand how to package a built RAG pipeline into a lightweight inference application.
- Understand how document preparation, retrieval quality, grounding, and source traceability work together in a practical RAG system.

# **3. Problem Statement**

## 3.1. Business Context

MeridianHealth Insurance is a mid-sized health insurance carrier headquartered in Hartford, Connecticut, and is licensed across 22 states. The company serves approximately **2.4 million covered lives across 47 active plan designs**, including individual, family, employer-sponsored, and supplemental plans.

A direct sales force of roughly **4,800 licensed representatives** handles live customer conversations. During these conversations, customers ask questions about benefits, waiting periods, deductibles, copays, exclusions, reimbursement, and claim processes. Some questions are simple and can be answered from an existing FAQ. However, many practical questions require the representative to look at more than one policy document and connect information from different sections.

Today, this process is largely manual. A representative may need to search a PDF, move to another document, read a table or condition, and then combine the findings into one answer. During a live customer conversation, this takes time and can make the process difficult to scale.

The business therefore wants an **AI-assisted policy knowledge system**. The system should help a representative ask a question in normal language, retrieve the relevant policy passages, and produce a clear answer based on those passages. The representative should also be able to see the source information used by the system.


## 3.2. Problem Framing


The business problem is that important policy information exists, but it is spread across several documents and is not always easy to find quickly. A representative may know what the customer is asking, but the answer may require reading a benefit table, checking an exclusion, and then confirming a condition in the terms and conditions.

A traditional FAQ cannot cover every possible way a customer may ask a question. Even when an FAQ contains related information, the representative may still need to open the original policy documents to verify the complete answer.

The required solution is therefore a system that can search the underlying policy documents using the **meaning of the question**, retrieve the most relevant passages, and give those passages to an LLM as context. The LLM should then generate a simple answer that is supported by the retrieved policy information.

This is the role of RAG in this case study: **use retrieval to find the evidence, then use generation to explain that evidence in a useful form.**

## 3.3. Key Business Challenges

The business problem can be broken into four practical challenges. Each challenge affects a different part of the RAG pipeline.


### 3.3.1. Finding the Right Information

A customer question may not use the same words as the policy document. It may also require information from a specific section, a table, an exclusion clause, or another policy document. A useful system therefore needs semantic retrieval, not only exact keyword matching.

For example, a representative may ask about the reimbursement available for a non-network service. The relevant information could be spread across the benefit rules, the reimbursement rules, and an exclusion or limitation section. The retrieval system must bring the right passages together.


### 3.3.2. Preserving Policy Meaning

Insurance documents contain headings, clauses, conditions, definitions, and tables. These structures carry business meaning. If a document is converted to plain text and split at poor boundaries, a retrieved chunk may lose the condition that explains when a benefit applies.

The document preparation stage therefore matters as much as the retrieval stage. We need to preserve useful structure before creating embeddings.


### 3.3.3. Grounding the Answer

A language model can generate fluent answers, but fluency alone is not enough for an insurance use case. The answer should be based on the MeridianHealth policy information retrieved for the question.

The RAG workflow addresses this by placing the retrieved policy passages inside the prompt given to the LLM. The model is then instructed to use that retrieved context as its evidence and to avoid inventing policy details when the evidence is not sufficient.


### 3.3.4. Traceability

A representative should not have to trust an answer without knowing where it came from. For this reason, each retrieved chunk keeps metadata such as the source file and chunk ID.

This metadata allows the application to show the supporting policy passages along with the generated answer. The goal is to make the system easier to inspect and use in a business workflow.


## 3.4. Objective


As an AI Engineer at MeridianHealth, your responsibility is to build a context-aware RAG solution that can help insurance representatives quickly find and understand information from MeridianHealth policy documents. Representatives often need to answer questions about benefits, coverage, waiting periods, exclusions, claim procedures, and other policy details, but finding the correct information across multiple PDF documents can take time. To solve this problem, you will design and build a RAG-based assistant that takes the MeridianHealth policy documents as its knowledge source, prepares and indexes the information for efficient retrieval, retrieves the most relevant policy content for a representative’s question, and uses an LLM to generate a clear answer based only on that retrieved information. The solution should also retain the source details of the retrieved content so that representatives can see where the answer came from. Finally, you will package this solution into a simple Streamlit application that can be used to ask policy questions and view the grounded responses.

# **4. Installing and Importing the Necessary Libraries**

In [1]:
# Install the necessary libraries with specified versions
!pip install docling==2.126.0 pandas==2.2.3 transformers==5.16.1 \
    langchain-huggingface==1.2.2 langchain-text-splitters==1.1.2 \
    langchain-community==0.4.2 faiss-cpu==1.15.0 \
    langchain-openai==1.6.1 sentence-transformers -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [6]:
# Core Python utilities
from pathlib import Path
import pandas as pd

# PDF/document understanding
from docling.document_converter import DocumentConverter

# Tokenization
from transformers import AutoTokenizer

# LangChain document and text-splitting utilities
from langchain_core.documents import Document as LCDocument
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings, vector database and LLM
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

# **5. Data Understanding**



## 5.1. Policy Documents

The RAG knowledge base consists of four MeridianHealth policy documents. Each document covers a different part of the insurance workflow, so together they form the knowledge base needed for multi-document retrieval.

| Document | What it contains | Why it matters for RAG |
|---|---|---|
| `MH_Plan_Benefits_Coverage_Guide.pdf` | Plan benefits, deductibles, copays, coinsurance, coverage tables | Supports benefit and cost-sharing questions |
| `MH_Claim_Exclusion_Reference.pdf` | Covered/non-covered services, exclusions, conditional coverage and limits | Supports exclusion and coverage-condition questions |
| `MH_Terms_and_Conditions.pdf` | Definitions, eligibility, waiting periods, renewal and termination provisions | Supports policy-rule and eligibility questions |
| `MH_Claim_Process_Reimbursement_Guide.pdf` | Claim filing, documentation, reimbursement, deadlines and appeals | Supports process and reimbursement questions |


In [7]:
# Define the root data path and list all policy document PDF file paths
root_path = "Data"

pdf_paths = [
    root_path + "/" + "MH_Plan_Benefits_Coverage_Guide.pdf",
    root_path + "/" + "MH_Claim_Exclusion_Reference.pdf",
    root_path + "/" + "MH_Terms_and_Conditions.pdf",
    root_path + "/" + "MH_Claim_Process_Reimbursement_Guide.pdf",
]

policy_files = pd.DataFrame({
    "Policy Document": [Path(path).name for path in pdf_paths]
})

policy_files

,Policy Document
0,MH_Plan_Benefits_Coverage_Guide.pdf
1,MH_Claim_Exclusion_Reference.pdf
2,MH_Terms_and_Conditions.pdf
3,MH_Claim_Process_Reimbursement_Guide.pdf


## 5.2. Supporting Reference: FAQ

The original FAQ is retained as a reference for the **past approach**. It is not part of the vector database used by the new RAG pipeline.

The FAQ is useful for understanding why a simple lookup-based system is helpful for known questions but becomes limited when users ask rephrased, multi-part, or cross-document questions.

This gives us a clear comparison:

**FAQ lookup → predefined answers**

versus

**RAG → semantic retrieval from the underlying policy documents → grounded generation**


The original FAQ is retained as a reference for the **past approach**. It is not part of the vector database used by the new RAG pipeline.

This helps us compare two approaches:

**FAQ lookup → predefined answers**

versus

**RAG → semantic retrieval from the underlying policy documents**

# **6. Past Approach**

In the past approach, MeridianHealth Insurance curated an FAQ based on the queries previously handled by the team. This serves as a good starting point, as repetitive queries can be handled effectively through simple lookups.

The FAQ is useful for **simple, repetitive queries**, but its coverage is limited for more complex real-world queries.

| Query Type | Example | FAQ Coverage | Limitation |
|---|---|---|---|
| **Direct** | “What is the deadline for submitting a reimbursement claim?” | FAQ-05 | Direct lookup works well. |
| **Direct** | “What are the copay amounts for a primary care visit under each plan tier?” | FAQ-01 | Direct lookup works well. |
| **Rephrased** | “My customer has diabetes and wants to know if they'll be covered right away on the Silver plan or if there's a waiting period.” | FAQ-02 | Requires semantic matching rather than exact lookup. |
| **Rephrased** | “A member went to a non-network ER after a car accident. Will they get hit with extra charges for being out of network?” | FAQ-06 | Requires understanding the query and mapping it to the relevant FAQ. |
| **Combination** | “A Gold plan member wants to get acupuncture for chronic back pain. Is it covered, does it need prior authorization, and how many sessions can they get?” | FAQ-12 + FAQ-08 (partial) + EXC-3.5 | FAQ provides only partial information; requires another policy document. |
| **Combination** | “A member on the Supplemental Shield plan was diagnosed with hypertension 8 months ago and is now hospitalized. What benefits can they claim and how do they file?” | FAQ-10 + FAQ-04/09 (partial) | Requires combining FAQ information with the Claim Process Guide. |
| **Outside FAQ** | “Can a 67-year-old retiree without any employer-sponsored coverage enroll in the Supplemental Shield plan?” | Not available | Requires Terms & Conditions, Section 2.3. |
| **Outside FAQ** | “How is the Maximum Allowable Amount calculated for out-of-network reimbursement, and what happens to charges that exceed it?” | Not available | Requires Terms & Conditions + Claim Process Guide. |

The FAQ handles **direct queries effectively**, but real-world queries often require **semantic matching, multi-document retrieval, information synthesis, or retrieval beyond the FAQ**. This highlights the need for a RAG-based approach.

The limitation of the FAQ approach is not that it cannot answer questions. It works well for known, repetitive queries.

The challenge is that real questions are often **rephrased, multi-part, cross-document, or outside the FAQ**. RAG addresses this by searching the underlying policy knowledge base semantically instead of depending only on exact FAQ matches.

# **7. RAG Workflow Overview**

Before implementing the code, keep the target architecture in mind.

The RAG workflow has two phases. First, we prepare and index the policy knowledge base. Second, at query time, we retrieve relevant chunks and use them as context for the LLM.

```text
Policy PDFs
   ↓
Document Parsing
   ↓
Chunking
   ↓
Embeddings
   ↓
FAISS Vector Database
   ↓
User Query
   ↓
Semantic Retrieval
   ↓
Retrieved Policy Context
   ↓
Prompt Augmentation
   ↓
LLM Generation
   ↓
Grounded Answer + Supporting Sources
   ↓
Streamlit Inference App
```

The rest of the notebook implements this workflow step by step.


# **8. Document Loading and Structuring**

Before we can retrieve policy information, we first need to turn the PDF files into usable text.

Policy PDFs can contain headings, paragraphs, numbered clauses, and tables. We use **Docling** to convert the PDFs into a structured document representation and export that representation as Markdown.

This gives the downstream chunking step better natural boundaries and helps preserve table structure.

## 8.1. Converting Policy PDFs with Docling

In [9]:
# Initialize the Docling document converter
converter = DocumentConverter()

# Adjust pdf_paths to point to the actual files in the root folder instead of the missing 'Data' directory
corrected_pdf_paths = [
    "MH_Plan_Benefits_Coverage_Guide.pdf",
    "MH_Claim_Exclusion_Reference.pdf",
    "MH_Terms_and_Conditions.pdf",
    "MH_Claim_Process_Reimbursement_Guide.pdf"
]

# Convert each policy PDF into a Docling document object
docs = {}

for path in corrected_pdf_paths:
    result = converter.convert(path)
    docs[path] = result.document
    print(f"{Path(path).name} - completed")

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

MH_Plan_Benefits_Coverage_Guide.pdf - completed
MH_Claim_Exclusion_Reference.pdf - completed
MH_Terms_and_Conditions.pdf - completed
MH_Claim_Process_Reimbursement_Guide.pdf - completed


## 8.2. Inspecting the Extracted Markdown

In [10]:
# Inspect one converted document to understand the extracted structure
sample_path, sample_doc = next(iter(docs.items()))

print(f"Document: {Path(sample_path).name}")
print(sample_doc.export_to_markdown()[:5000])

Document: MH_Plan_Benefits_Coverage_Guide.pdf



Look at the output before chunking. We want to verify that headings, paragraphs, lists, and tables are represented in a way that can be meaningfully split and embedded.

This inspection step is useful because **document parsing quality directly affects downstream retrieval quality**.

## 8.3. Inspecting Extracted Tables

In [11]:
# Display tables extracted from the policy documents
for path, doc in docs.items():
    for item, _ in doc.iterate_items():
        if getattr(item, "label", "") == "table":
            print(f"\n--- Table from: {Path(path).name} ---")
            print(item.export_to_markdown(doc=doc)[:3000])


--- Table from: MH_Plan_Benefits_Coverage_Guide.pdf ---


--- Table from: MH_Plan_Benefits_Coverage_Guide.pdf ---


--- Table from: MH_Plan_Benefits_Coverage_Guide.pdf ---


--- Table from: MH_Claim_Exclusion_Reference.pdf ---


--- Table from: MH_Claim_Process_Reimbursement_Guide.pdf ---


--- Table from: MH_Claim_Process_Reimbursement_Guide.pdf ---



Tables are especially important in an insurance use case because many policy questions depend on structured relationships, such as **plan tier → benefit → amount**.

Preserving this structure makes it easier for retrieved chunks to retain the meaning of the original policy.

# **9. Chunking**

The full policy PDFs are too large to send to the LLM for every question. Instead, we divide them into smaller pieces called **chunks**.

A good chunk should contain enough information to answer a question while remaining small enough for the embedding model and retrieval workflow. Chunking also helps the vector database index smaller, more focused pieces of information.

For these policy documents, we use a recursive, token-aware strategy. The splitter tries to keep larger natural units together and only moves to smaller units when the current piece is still too large.


## 9.1. Chunking and Chunk Overlap


Chunk overlap means that some content is repeated between consecutive chunks. This creates a small amount of redundancy, but it can help preserve meaning when a useful statement crosses a chunk boundary.

In this notebook we use a **50-token overlap**.

## 9.2. Choosing the Embedding Model

Embedding models are not interchangeable. Their behaviour can depend on the domain, the language used in the data, and the type of retrieval task.

Because these documents describe medical insurance policies, we use the domain-oriented embedding model:

**`surajvbangera/mediclaim_embedding`**

We also use its tokenizer when defining the chunk size so that chunking and embedding use the same token definition.


In [12]:
# Domain-specific embedding model
EMBED_MODEL = "surajvbangera/mediclaim_embedding"

# Load the tokenizer used by the embedding model
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)

print(f"Embedding model: {EMBED_MODEL}")
print(f"Model max tokens: {tokenizer.model_max_length}")

Embedding model: surajvbangera/mediclaim_embedding
Model max tokens: 512


The tokenizer reports a maximum length of **512 tokens** for this model, so we use that as the upper chunk-size target.

Using the same tokenizer for chunking is important because different tokenizers can split the same text differently.

In [13]:
# Inspect how the tokenizer splits a policy-related sentence
sample = "MeridianHealth offers four ACA-compliant metal tiers."

tokens = tokenizer.tokenize(sample)

print(f"Sample: {sample}")
print(f"Token count: {len(tokens)}")
print(f"Tokens: {tokens}")

Sample: MeridianHealth offers four ACA-compliant metal tiers.
Token count: 13
Tokens: ['meridian', '##hea', '##lth', 'offers', 'four', 'ac', '##a', '-', 'compliant', 'metal', 'tier', '##s', '.']


## 9.3. Defining the RecursiveCharacterTextSplitter

We use `RecursiveCharacterTextSplitter` with the embedding model's tokenizer.

The splitter follows a priority order. It first tries larger natural boundaries such as paragraphs. If a resulting piece is still too large, it tries a smaller boundary such as a line break, then a space, and finally a character-level split as a last resort.

This strategy is useful for policy documents because we want to preserve meaningful structure whenever the chunk-size limit allows it.

In [14]:
# Token-aware recursive chunking
splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=512,
    chunk_overlap=50,
    separators=["\n\n", "\n", "##", ". ", " ", ""],
)

**Why These Parameters?**
- **512 tokens:** Keeps chunks within the embedding model's advertised input limit.
- **50-token overlap:** Preserves continuity when a relevant statement crosses a chunk boundary.
- **Structure-aware separators:** Gives paragraphs and line/section boundaries a chance to define the chunk before the splitter falls back to smaller units.

In [40]:
from langchain_core.documents import Document as LCDocument
from pathlib import Path
from docling.document_converter import DocumentConverter

# Initialize converter if not present
if 'converter' not in globals():
    converter = DocumentConverter()

raw_docs = []

# Safely convert documents and extract non-empty markdown
for path in corrected_pdf_paths:
    # Retrieve or convert document
    if path in docs:
        doc_obj = docs[path]
    else:
        doc_obj = converter.convert(path).document
        docs[path] = doc_obj

    # Export to markdown text
    extracted_text = doc_obj.export_to_markdown()

    # Ensure we get content; fallback to standard string representation if markdown is empty
    if not extracted_text.strip():
        extracted_text = str(doc_obj)

    raw_docs.append(
        LCDocument(
            page_content=extracted_text,
            metadata={"source": path},
        )
    )

# Split the policy documents into retrieval-friendly chunks
lc_docs = splitter.split_documents(raw_docs)

# Add simple traceability metadata to every chunk
for i, chunk in enumerate(lc_docs):
    source_name = Path(chunk.metadata["source"]).name
    chunk.metadata["source_file"] = source_name
    chunk.metadata["chunk_id"] = f"chunk_{i:03d}"

print(f"Total chunks created: {len(lc_docs)}")

Total chunks created: 9


Each chunk now carries:

- the chunk text,
- the original source path,
- a readable source filename,
- a unique chunk ID.

This metadata will be used later when we show the supporting sources in the RAG answer and in the Streamlit app.

In [16]:
# Inspect a few chunks to check whether the boundaries are sensible
for chunk in lc_docs[:5]:
    print(f"\n--- {chunk.metadata['chunk_id']} | {chunk.metadata['source_file']} ---")
    print(chunk.page_content[:1000])

In [ ]:
# Inspect chunk sizes using the same tokenizer as the embedding model
chunk_token_counts = [
    len(tokenizer.encode(chunk.page_content, add_special_tokens=False))
    for chunk in lc_docs
]

pd.Series(chunk_token_counts).describe()

,0
count,22.000000
mean,451.045455
std,74.350186
min,223.000000
25%,423.500000
50%,484.500000
75%,498.000000
max,509.000000


The chunk-size summary gives a quick validation that the splitter is producing chunks within the intended token budget and helps us understand the actual chunk-size distribution.

# **10. Vector Database**

Once the documents have been chunked, the next step is to convert every chunk into a dense vector and store those vectors in a searchable index.

The flow is:

**Chunk → Embedding → Vector Database**

## 10.1. Loading the Embedding Model

In [17]:
# Load the same embedding model used to choose the tokenizer
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print(f"Embedding model loaded: {EMBED_MODEL}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded: surajvbangera/mediclaim_embedding


## 10.2. Creating the FAISS Vector Database

We use **FAISS** as the vector store because it provides a lightweight local index for dense-vector similarity search.

For this MLS, FAISS keeps the architecture simple: learners can create the vector database locally and then reuse the saved index in a separate inference application without running a database server.

In [30]:
# Validate that the document chunk list is not empty before building the index
if not lc_docs:
    raise ValueError(
        "The 'lc_docs' list is empty. Please verify that the previous document processing "
        "and chunking cells (like cell a9f8d96e) executed successfully and extracted non-empty text."
    )

# Build the FAISS vector store from the policy chunks
vectorstore = FAISS.from_documents(
    documents=lc_docs,
    embedding=embeddings,
)

# Persist the vector index locally for reuse by the Streamlit app
vectorstore.save_local("faiss_index")

print(f"Vectors stored in FAISS: {vectorstore.index.ntotal}")

Vectors stored in FAISS: 9


The saved `faiss_index` folder is now the reusable **searchable knowledge base**.

The next step is to ask the vector store for the policy chunks that are most relevant to a user query.

# **11. Retrieval System**

At inference time, the user's question is compared with the vectors stored in the FAISS index.

The vector store returns the top-`k` chunks that are most similar to the query. These chunks become the context that is passed to the LLM.

We start with **`k = 3`** so that a query can retrieve more than one relevant clause when the answer spans multiple parts of the policy.

In [31]:
# Quick retrieval test
query = "What is the specialist visit copay for the Gold 80 plan?"

results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"\n--- Retrieved Chunk {i} | {doc.metadata['source_file']} | {doc.metadata['chunk_id']} ---")
    print(doc.page_content[:1500])


--- Retrieved Chunk 1 | MH_Plan_Benefits_Coverage_Guide.pdf | chunk_000 ---
schema_name='DoclingDocument' version='1.10.0' name='MH_Plan_Benefits_Coverage_Guide' origin=DocumentOrigin(mimetype='application/pdf', binary_hash=15059095495081810346, filename='MH_Plan_Benefits_Coverage_Guide.pdf', uri=None) furniture=GroupItem(self_ref='#/furniture', parent=None, children=[], content_layer=<ContentLayer.FURNITURE: 'furniture'>, meta=None, name='_root_', label=<GroupLabel.UNSPECIFIED: 'unspecified'>) body=GroupItem(self_ref='#/body', parent=None, children=[RefItem(cref='#/tables/0'), RefItem(cref='#/tables/1'), RefItem(cref='#/tables/2')], content_layer=<ContentLayer.BODY: 'body'>, meta=None, name='_root_', label=<GroupLabel.UNSPECIFIED: 'unspecified'>) groups=[] texts=[] pictures=[] tables=[TableItem(self_ref='#/tables/0', parent=RefItem(cref='#/body'), children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, label=<DocItemLabel.TABLE: 'table'>, prov=[ProvenanceItem(page_no=2, bb

## 11.1. What Should We Inspect Here?



Before moving to generation, check whether the retrieved chunks actually contain the information needed for the question.

This is an important RAG debugging principle:

> **If retrieval is wrong, a good LLM prompt cannot reliably recover the correct answer.**

## 11.2. Building a Reusable Retrieval Function

In [32]:
def retrieve(query: str, k: int = 3, vectorstore=None) -> list:
    # Retrieve the top-k policy chunks for a user query.
    if vectorstore is None:
        raise ValueError("vectorstore must be provided.")

    return vectorstore.similarity_search(query, k=k)

In [33]:
# Test the reusable retrieval function
query = "What are the copay amounts for a primary care visit under each plan tier?"

retrieved_docs = retrieve(
    query=query,
    k=3,
    vectorstore=vectorstore,
)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Result {i} ---")
    print(f"Source: {doc.metadata['source_file']}")
    print(f"Chunk ID: {doc.metadata['chunk_id']}")
    print(doc.page_content[:1200])


--- Result 1 ---
Source: MH_Terms_and_Conditions.pdf
Chunk ID: chunk_005
schema_name='DoclingDocument' version='1.10.0' name='MH_Terms_and_Conditions' origin=DocumentOrigin(mimetype='application/pdf', binary_hash=3868773049883530326, filename='MH_Terms_and_Conditions.pdf', uri=None) furniture=GroupItem(self_ref='#/furniture', parent=None, children=[], content_layer=<ContentLayer.FURNITURE: 'furniture'>, meta=None, name='_root_', label=<GroupLabel.UNSPECIFIED: 'unspecified'>) body=GroupItem(self_ref='#/body', parent=None, children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, name='_root_', label=<GroupLabel.UNSPECIFIED: 'unspecified'>) groups=[] texts=[] pictures=[] tables=[] key_value_items=[] form_items=[] field_regions=[] field_items=[] pages={1: PageItem(size=Size(width=612.0, height=792.0), image=None, page_no=1), 2: PageItem(size=Size(width=612.0, height=792.0), image=None, page_no=2), 3: PageItem(size=Size(width=612.0, height=792.0), image=None, page_no=3), 4: PageI

Returning the original `Document` objects instead of converting them immediately to plain strings keeps the text and metadata together.

That is useful because the generation stage needs the text, while the application later needs the metadata for source traceability.

# **12. Generation System**

## 12.1. Configuration and LLM Setup

The RAG system needs a generator LLM to turn retrieved policy context into a natural-language answer.

For local development and the Streamlit application, use a private `config.json` file containing the API credentials and endpoint. Keep this file private and do not commit it to source control.

The same `config.json` file will be used by both the notebook and the Streamlit app, so the configuration remains consistent across the development and inference steps.


**12.1.1. `config.json` template**

Create a local file named **`config.json`** with:

```json
{
    "OPENAI_API_KEY": "your-openai-key",
    "OPENAI_API_BASE": "https://aibe.mygreatlearning.com/openai/v1"
}
```

Replace the placeholder key with the credential provided for your learning environment.

> **Security:** Never hard-code a real API key into the notebook or commit `config.json` to Git.


In [42]:
import json

# Load API configuration from config.json
# Ensure config.json is in the same directory as this notebook.
CONFIG_FILE_PATH = 'config.json'

with open(CONFIG_FILE_PATH, 'r') as f:
    config = json.load(f)
OPENAI_API_KEY = config.get("OPENAI_API_KEY")
OPENAI_API_BASE = config.get("OPENAI_API_BASE")

# Validate that the keys are present after loading
if not OPENAI_API_KEY or not OPENAI_API_BASE:
    raise ValueError(f"Missing OPENAI_API_KEY or OPENAI_API_BASE in {CONFIG_FILE_PATH}. Please ensure these keys are present in your config.json.")

In [43]:
# Generation model configuration
MODEL = "gpt-4o-mini"
TEMPERATURE = 0
TOP_P = 1
MAX_TOKENS = 512

# Initialize the generator LLM once and reuse it
llm = ChatOpenAI(
    model=MODEL,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    max_tokens=MAX_TOKENS,
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

print(f"Generator model ready: {MODEL}")

Generator model ready: gpt-4o-mini


In [44]:
# Verify that the LLM connection is working
TEST_QUERY_LLM = "Say 'LLM is ready' and nothing else."

response = llm.invoke(TEST_QUERY_LLM)

print(response.content)

LLM is ready.


## 12.2. Defining the Generation Prompt

The retrieved chunks are the **evidence** given to the LLM.

The prompt should make that relationship explicit:

**User question + Retrieved policy context → Grounded answer**

The model is instructed not to invent policy details and to clearly say when the retrieved context is insufficient.

In [45]:
def format_context(retrieved_docs: list) -> str:
    # Format retrieved Documents as a context block with source metadata.
    context_blocks = []

    for i, doc in enumerate(retrieved_docs, start=1):
        source = doc.metadata.get("source_file", "Unknown source")
        chunk_id = doc.metadata.get("chunk_id", f"chunk_{i}")

        context_blocks.append(
            f"[Source {i} | {source} | {chunk_id}]\n{doc.page_content}"
        )

    return "\n\n".join(context_blocks)

In [46]:
def generate(query: str, retrieved_docs: list, model) -> str:
    # Generate a grounded answer using only the retrieved policy context.
    context = format_context(retrieved_docs)

    prompt = f"""
You are an AI assistant helping MeridianHealth insurance representatives
answer policy-related questions.

User Query:
{query}

Retrieved Policy Context:
{context}

Instructions:
1. Answer using only the retrieved policy context.
2. Do not use outside knowledge, assumptions, or invented policy details.
3. Include important conditions, exclusions, limits, waiting periods, and exceptions
   when they are necessary to answer the question.
4. For multi-part questions, answer each part clearly.
5. Keep the response concise and easy for a representative to understand.
6. If the retrieved context is not sufficient, say:
   "The available policy documents do not contain enough information to answer this question."
7. If the retrieved context contains conflicting policy information, clearly flag
   the conflict instead of choosing an unsupported answer.

Answer:
"""

    response = model.invoke(prompt)
    return response.content

# **13. Retrieval-Augmented Generation System**

We now combine the two components:

**R — Retrieve relevant policy chunks**  
**A — Augment the LLM prompt with those chunks**  
**G — Generate the answer**

So the complete flow becomes:

`User Query → Retrieve → Augment Context → Generate`

In [47]:
def rag(query: str, k: int = 3, vectorstore=None, model=None):
    # Run the complete Retrieval-Augmented Generation workflow.
    if vectorstore is None:
        raise ValueError("vectorstore must be provided.")

    if model is None:
        raise ValueError("model must be provided.")

    # 1. Retrieve relevant policy chunks
    retrieved_docs = retrieve(
        query=query,
        k=k,
        vectorstore=vectorstore,
    )

    # 2. Augment the prompt and generate the grounded answer
    answer = generate(
        query=query,
        retrieved_docs=retrieved_docs,
        model=model,
    )

    return answer, retrieved_docs

## 13.1. Testing the RAG Pipeline

In [48]:
query = "What are the copay amounts for a primary care visit under each plan tier?"

response, retrieved_docs = rag(
    query=query,
    k=3,
    vectorstore=vectorstore,
    model=llm,
)

print("Answer:")
print(response)

Answer:
The available policy documents do not contain enough information to answer this question.


In [49]:
# Display the sources that supported the answer
for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\n--- Source {i} ---")
    print(f"Document: {doc.metadata['source_file']}")
    print(f"Chunk ID: {doc.metadata['chunk_id']}")
    print(doc.page_content[:1500])


--- Source 1 ---
Document: MH_Terms_and_Conditions.pdf
Chunk ID: chunk_005
schema_name='DoclingDocument' version='1.10.0' name='MH_Terms_and_Conditions' origin=DocumentOrigin(mimetype='application/pdf', binary_hash=3868773049883530326, filename='MH_Terms_and_Conditions.pdf', uri=None) furniture=GroupItem(self_ref='#/furniture', parent=None, children=[], content_layer=<ContentLayer.FURNITURE: 'furniture'>, meta=None, name='_root_', label=<GroupLabel.UNSPECIFIED: 'unspecified'>) body=GroupItem(self_ref='#/body', parent=None, children=[], content_layer=<ContentLayer.BODY: 'body'>, meta=None, name='_root_', label=<GroupLabel.UNSPECIFIED: 'unspecified'>) groups=[] texts=[] pictures=[] tables=[] key_value_items=[] form_items=[] field_regions=[] field_items=[] pages={1: PageItem(size=Size(width=612.0, height=792.0), image=None, page_no=1), 2: PageItem(size=Size(width=612.0, height=792.0), image=None, page_no=2), 3: PageItem(size=Size(width=612.0, height=792.0), image=None, page_no=3), 4: Pag

At this point, the notebook has completed the core RAG workflow.

The answer is generated from retrieved policy content, and the same retrieved documents are available for source inspection.

## 13.2. Reloading the Vector Database

The vector database does not need to be rebuilt every time the inference application starts.

Once `faiss_index` has been created and saved, another Python program can load it using the same embedding model.

In [50]:
# Example: reload the persisted FAISS index
reload_embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

reloaded_vectorstore = FAISS.load_local(
    "faiss_index",
    reload_embeddings,
    allow_dangerous_deserialization=True,
)

print(f"Reloaded vectors: {reloaded_vectorstore.index.ntotal}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Reloaded vectors: 9


This persistence step is what allows the Streamlit app to behave as an **inference application** rather than rebuilding the knowledge base every time the user opens the app.

# **14. Building and Running the Streamlit Inference App in Local VS Code**


The notebook now packages the reusable RAG components into a simple inference application.

The app will:

1. Load the API configuration from `config.json`.
2. Load the saved `faiss_index`.
3. Load the same embedding model used during indexing.
4. Accept a representative's policy question.
5. Retrieve relevant policy chunks.
6. Generate a grounded answer using the retrieved context.
7. Display the supporting source passages.

The application does **not** rebuild the policy documents or the vector database. Those steps have already been completed in the notebook.

## 14.1. Writing `app.py` with Colab Magic


In [51]:
%%writefile app.py
import json
from pathlib import Path

import streamlit as st
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI


# Keep all application files relative to the folder containing app.py
BASE_DIR = Path(__file__).resolve().parent

CONFIG_FILE_PATH = BASE_DIR / "config.json"
FAISS_INDEX_PATH = BASE_DIR / "faiss_index"

MODEL = "gpt-4o-mini"
EMBED_MODEL = "surajvbangera/mediclaim_embedding"
TOP_K = 3


def load_config():
    """Load API configuration from config.json."""
    if not CONFIG_FILE_PATH.exists():
        raise FileNotFoundError(
            f"Configuration file not found: {CONFIG_FILE_PATH}"
        )

    with open(CONFIG_FILE_PATH, "r", encoding="utf-8") as f:
        config = json.load(f)

    openai_api_key = config.get("OPENAI_API_KEY")
    openai_api_base = config.get("OPENAI_API_BASE")

    if not openai_api_key or not openai_api_base:
        raise ValueError(
            "config.json must contain OPENAI_API_KEY and OPENAI_API_BASE."
        )

    return openai_api_key, openai_api_base


@st.cache_resource
def load_resources():
    """Load the embedding model, FAISS index, and generator LLM once."""
    openai_api_key, openai_api_base = load_config()

    embeddings = HuggingFaceEmbeddings(
        model_name=EMBED_MODEL,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )

    vectorstore = FAISS.load_local(
        str(FAISS_INDEX_PATH),
        embeddings,
        allow_dangerous_deserialization=True,
    )

    llm = ChatOpenAI(
        model=MODEL,
        temperature=0,
        top_p=1,
        max_tokens=512,
        api_key=openai_api_key,
        base_url=openai_api_base,
    )

    return vectorstore, llm


def build_context(docs):
    """Format retrieved chunks with source metadata for the LLM."""
    blocks = []

    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get(
            "source_file",
            Path(doc.metadata.get("source", "Unknown source")).name,
        )
        chunk_id = doc.metadata.get("chunk_id", f"chunk_{i}")

        blocks.append(
            f"[Source {i} | {source} | {chunk_id}]\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(blocks)


def answer_query(query, vectorstore, llm):
    """Retrieve relevant policy chunks and generate a grounded answer."""
    retrieved_docs = vectorstore.similarity_search(query, k=TOP_K)

    context = build_context(retrieved_docs)

    prompt = f"""
You are an AI assistant helping MeridianHealth insurance representatives
answer policy-related questions.

User Query:
{query}

Retrieved Policy Context:
{context}

Instructions:
1. Answer using only the retrieved policy context.
2. Do not use outside knowledge or invent policy details.
3. Include important conditions, exclusions, limitations, and waiting periods
   when relevant.
4. Answer every part of a multi-part question.
5. Keep the response concise and clear.
6. If the retrieved context is insufficient, say:
   "The available policy documents do not contain enough information to answer this question."
7. If the retrieved context contains conflicting information, clearly flag it.

Answer:
"""

    response = llm.invoke(prompt)

    return response.content, retrieved_docs


st.set_page_config(
    page_title="MeridianHealth RAG Assistant",
    page_icon="🩺",
    layout="wide",
)

st.title("MeridianHealth RAG Assistant")
st.caption("Context-aware answers from MeridianHealth policy documents")


try:
    vectorstore, llm = load_resources()
except Exception as exc:
    st.error(
        "Could not load the RAG system. Make sure `config.json` and the "
        "`faiss_index` folder are present in the same project folder."
    )
    st.caption(f"Details: {exc}")
    st.stop()


query = st.chat_input("Ask a policy question...")

if query:
    with st.spinner("Searching policy documents and generating an answer..."):
        answer, retrieved_docs = answer_query(query, vectorstore, llm)

    st.subheader("Answer")
    st.write(answer)

    st.subheader("Supporting Policy Sources")

    for i, doc in enumerate(retrieved_docs, start=1):
        source = doc.metadata.get(
            "source_file",
            Path(doc.metadata.get("source", "Unknown source")).name,
        )
        chunk_id = doc.metadata.get("chunk_id", f"chunk_{i}")

        with st.expander(f"Source {i}: {source} | {chunk_id}"):
            st.write(doc.page_content)

Writing app.py


The code above creates the complete `app.py` file in the current notebook environment.

The application is designed for **local execution in VS Code**. It uses the same `config.json`, embedding model, and persisted `faiss_index` created earlier in the notebook.

We will now package the files needed by the application and move them from the Colab environment to a local project folder.

## 14.2. Preparing the App and Vector Database for Download

The Streamlit application needs two artifacts created in this notebook:

- `app.py` — the inference application.
- `faiss_index/` — the persisted FAISS vector database.

We will package these into one ZIP file so they can be downloaded from Colab and copied into a local project folder.

> **Security note:** We do not package `config.json` with the application bundle because it contains an API credential. The `config.json` file will be created locally in Section **14.8**.

In [52]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

APP_FILE = Path("/content/app.py")
FAISS_DIR = Path("/content/faiss_index")
BUNDLE_FILE = Path("/content/meridianhealth_rag_app_bundle.zip")

if not APP_FILE.exists():
    raise FileNotFoundError("app.py was not found in /content.")

if not (FAISS_DIR / "index.faiss").exists() or not (FAISS_DIR / "index.pkl").exists():
    raise FileNotFoundError(
        "The expected FAISS files were not found in /content/faiss_index."
    )

# Create a fresh bundle containing app.py and the saved FAISS vector database.
if BUNDLE_FILE.exists():
    BUNDLE_FILE.unlink()

with ZipFile(BUNDLE_FILE, "w", compression=ZIP_DEFLATED) as zip_file:
    zip_file.write(APP_FILE, arcname="app.py")

    for file_path in FAISS_DIR.rglob("*"):
        if file_path.is_file():
            zip_file.write(
                file_path,
                arcname=str(Path("faiss_index") / file_path.relative_to(FAISS_DIR)),
            )

print(f"Created bundle: {BUNDLE_FILE}")
print("Bundle contents:")
with ZipFile(BUNDLE_FILE, "r") as zip_file:
    for name in zip_file.namelist():
        print(f" - {name}")


Created bundle: /content/meridianhealth_rag_app_bundle.zip
Bundle contents:
 - app.py
 - faiss_index/index.pkl
 - faiss_index/index.faiss


## 14.3. Downloading the App Bundle from Colab

Run the following cell in Google Colab to download the ZIP file to your computer.

In [54]:
from google.colab import files

files.download("/content/meridianhealth_rag_app.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Your browser will download:

`meridianhealth_rag_app.zip`

The ZIP contains:

```text
meridianhealth_rag_app/
├── app.py
└── faiss_index/
    ├── index.faiss
    └── index.pkl
```

Keep the ZIP file on your computer and extract it in the local project folder described below.

## 14.4. Creating the Local Project Folder


Create a folder on your computer for the MeridianHealth RAG application.

A simple project structure is:

```text
meridianhealth-rag/
├── app.py
├── config.json
└── faiss_index/
    ├── index.faiss
    └── index.pkl
```

After downloading the ZIP file from Colab:

1. Create a new folder named **`meridianhealth-rag`**.
2. Extract the downloaded ZIP file.
3. Move `app.py` and the `faiss_index` folder into the `meridianhealth-rag` folder.
4. Do **not** move or rename the files inside `faiss_index`.

At this point, `app.py` and the vector database are together in the same local project directory.

## 14.5. Opening the Project in VS Code


Open **VS Code** and open the `meridianhealth-rag` folder.

You can use:

**VS Code → File → Open Folder → `meridianhealth-rag`**

The VS Code Explorer should show:

```text
meridianhealth-rag
├── app.py
├── config.json
└── faiss_index
    ├── index.faiss
    └── index.pkl
```

Open a new terminal in VS Code using:

**Terminal → New Terminal**

The terminal should start in the `meridianhealth-rag` project folder.

## 14.6. Creating a Virtual Environment


A virtual environment keeps the application's Python packages isolated from other projects.

In the VS Code terminal, run:

```bash
python -m venv .venv
```

Activate the environment.

**Windows:**

```bash
.venv\Scripts\activate
```

**macOS/Linux:**

```bash
source .venv/bin/activate
```

After activation, you should see `(.venv)` at the beginning of the terminal prompt.

## 14.7. Installing the Required Packages


Install the packages used by `app.py`:

```bash
pip install streamlit langchain-huggingface langchain-community langchain-openai faiss-cpu transformers sentence-transformers
```

The app does not rebuild the documents or embeddings. It only loads the persisted FAISS index, embeds the incoming query, retrieves relevant chunks, and sends the retrieved context to the generator LLM.

## 14.8. Creating `config.json`

Create a new file named **`config.json`** in the same folder as `app.py`.

Use this template:

```json
{
    "OPENAI_API_KEY": "your-openai-key",
    "OPENAI_API_BASE": "https://aibe.mygreatlearning.com/openai/v1"
}
```

Replace the placeholder value with the credential provided for your learning environment.

> **Security:** Keep `config.json` private. Do not commit it to GitHub or share it publicly.

## 14.9. Running the Streamlit Application


In the VS Code terminal, make sure you are inside the `meridianhealth-rag` folder and the virtual environment is active.

Run:

```bash
streamlit run app.py
```

Streamlit will start a local web server and normally open the MeridianHealth RAG Assistant in your browser.

The application will:

`User Question → Retrieve Policy Chunks → Add Context → Generate Answer → Show Supporting Sources`

The `faiss_index` is loaded from the local project folder, so the application does not need to rebuild the vector database every time it starts.

## 14.10. Testing the Application with Policy Questions



After the dashboard opens, try the following questions. These cover different parts of the MeridianHealth policy knowledge base such as reimbursement, exclusions, waiting periods, plan benefits, and prior authorization.

### Test Question 1 — Out-of-Network Reimbursement

```text
What steps does a member need to follow to file a reimbursement claim for out-of-network services?
```

### Test Question 2 — Plan Exclusion

```text
Is bariatric surgery covered under the Bronze plan?
```

### Test Question 3 — Waiting Period

```text
What is the waiting period for illness-related claims under the Supplemental Shield plan, and which benefits does it affect?
```

### Test Question 4 — Plan Comparison

```text
What is the difference in annual deductible between the Bronze and Platinum plans for an individual member?
```

### Test Question 5 — Prior Authorization and Coverage Limit

```text
Does acupuncture require prior authorization on the Gold plan, and how many sessions are covered?
```

For each question, inspect both the **generated answer** and the **supporting policy sources** shown by the application.

## 14.11. Important Notes



Keep the following points in mind when running the application locally:

**1. Keep the project structure unchanged.**  
`app.py`, `config.json`, and the `faiss_index` folder should remain in the same project folder.

**2. Keep the FAISS files together.**  
Both `index.faiss` and `index.pkl` are required to reload the persisted vector database.

**3. Do not share the API credential.**  
The API key belongs in `config.json`, and this file should remain private.

**4. Do not rebuild the vector database in the Streamlit app.**  
The notebook has already completed document processing, chunking, embedding, and FAISS indexing.

**5. Understand the deployment flow.**

```text
Policy PDFs
    ↓
Document Preparation
    ↓
Chunking
    ↓
Embeddings
    ↓
FAISS Vector Database
    ↓
Download app.py + faiss_index
    ↓
Local VS Code Project
    ↓
Streamlit Inference App
    ↓
Query → Retrieval → Grounded Generation
```

This completes the transition from a notebook-based RAG pipeline to a simple local inference application.


# **15. Business Insights and Recommendations**


## 15.1. Business Insights

### 15.1.1. Policy knowledge is distributed across documents

The case study shows that the information needed by a representative is not always stored in one place. A single question may require a benefit rule, an exclusion, and a claim or reimbursement condition. This is why the solution needs retrieval across the full policy knowledge base rather than a small predefined FAQ.

### 15.1.2. Semantic retrieval helps match real customer language to policy language

Representatives and customers may use different words for the same concept. For example, a customer may ask about their share of the cost while the policy document may use terms such as copay, coinsurance, or deductible. Embeddings allow the system to search by meaning instead of depending only on exact keyword matches.

### 15.1.3. Document preparation directly affects the usefulness of the answer

Policy documents contain headings, clauses, definitions, and tables. Keeping these structures as much as possible during document processing and then creating sensible chunks helps the retriever bring back useful evidence for the LLM.

### 15.1.4. Source traceability is important in a business workflow

The application shows the source document and chunk ID for the passages used in the response. This gives the representative a way to inspect the supporting policy content instead of relying only on the generated answer.

### 15.1.5. The vector database becomes a reusable knowledge layer

The FAISS index is created once from the policy documents and then reused by the application. This separates the heavier document-preparation work from the faster query-time workflow.

## 15.2. Business Recommendations

### 15.2.1. Add richer policy metadata

For a production system, store metadata such as plan name, state, policy type, effective date, and document version. This can help the application retrieve the correct policy content for a specific customer or representative query.

### 15.2.2. Keep the knowledge base aligned with policy changes

Insurance policies can change over time. The index should therefore be refreshed whenever approved policy documents, endorsements, or coverage rules are updated. Old document versions should be managed carefully so that outdated information is not retrieved for a current query.

### 15.2.3. Support multi-document questions explicitly

Business users should be able to ask questions that combine benefits, exclusions, eligibility, and reimbursement information. The retrieval design should therefore support retrieving relevant evidence from multiple policy documents rather than assuming that one document contains the complete answer.

### 15.2.4. Preserve source information in the user experience

The representative-facing application should continue to show the supporting document and passage used for each answer. This is useful when the representative needs to verify a policy rule during a live customer interaction.

### 15.2.5. Add a human escalation path for insufficient or conflicting evidence

When the available policy context is incomplete or conflicting, the system should clearly flag the issue and allow the representative to verify the policy or escalate the case. The assistant should support the representative rather than silently filling missing policy details.

### 15.2.6. Start with decision support and expand gradually

A practical rollout is to begin with representative-facing policy search and answer support. Once the workflow is stable, the same knowledge layer can be extended to additional approved use cases such as claims support, internal policy search, and customer-service assistance.